This tutorial walks through designing nanobody binders with BoltzGen on
the OpenProtein platform, then validating designs by:

- Inverse folding with PoET-2 to propose sequences for the designed
  nanobody backbones
- Structure prediction with Boltz-2 for the target–nanobody complex

We illustrate how to take an example from the official BoltzGen repo and
create a query to run on our platform.

# Prerequisites

For this tutorial, you will need your OpenProtein python session for
accessing the models available on our platform and manipulating job
results, so make sure you have your credentials setup!

In [ ]:
import openprotein
session = openprotein.connect()
session


# Target selection (Penguinpox cGAMP PDE)

We will use the penguinpox virus, which is also used in the BoltzGen
examples. Let's download the target structure file and load it as a
`Protein`.

In [ ]:
from openprotein import Protein
import requests
from pathlib import Path
from molviewspec import create_builder

DATA_DIR = Path("data/penguinpox")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Target structure (mmCIF)
target_url = "https://raw.githubusercontent.com/HannesStark/boltzgen/main/example/nanobody_against_penguinpox/9bkq-assembly2.cif"
TARGET_NAME = "9bkq-assembly2.cif"
TARGET_PATH = DATA_DIR / TARGET_NAME
if not TARGET_PATH.exists():
    TARGET_PATH.write_bytes(requests.get(target_url).content)

target_protein = Protein.from_filepath(path=TARGET_PATH, chain_id="B")
print("target sequence:", target_protein.sequence)
print("target coordinates shape:", target_protein.coordinates.shape)
print("target plddt shape:", target_protein.plddt.shape)
print("target name:", target_protein.name)


Now let's do a quick visualization of our target.

In [ ]:
# Quick visualization of the target
builder = create_builder()
vis = (builder.download(url="target.cif")
    .parse(format="mmcif")
    .model_structure()
    .component()
    .representation())
builder.molstar_notebook(data={"target.cif": target_protein.make_cif_string()}, width=500, height=400)


# Designing the nanobody scaffold

Next, we will want to design the scaffold for design with the target
virus. As a reference, we will use one of the scaffolds from the
BoltzGen examples as well. This is the 7EOW, which is also part of the
nanobody (VHH) scaffold. First let's retrieve the specification that
shows how the scaffold is built.

In [ ]:
scaffold_spec_url = "https://raw.githubusercontent.com/HannesStark/boltzgen/main/example/nanobody_scaffolds/7eow.yaml"
print(requests.get(scaffold_spec_url).text)


This specification from BoltzGen describes how the scaffold protein is
prepared for use in a design workflow. There's no need to actually learn
the specification to create designs on our platform, but it is useful as
a reference.

In particular, we can see that we want to:

- Use chain B in the structure
- Design residues 26 - 34, 52 - 59, and 98 - 118
- Set the visibility groups for the above residues to 0 and 2 for
  everything else
- Remove residues 26 - 28, 52 - 54, and 98 - 104
- Insert residues at each of the removed positions.

Resetting the indices is not necessary, since this is implicitly done
when we encode and upload our query.

Note that the order of operations is important since the residue indices
are shifted during deletion and insertion.

## Residue groups

It is worth spending some time to understand more about residue groups
and visibility, especially since it affects the design results.

The core idea is that everything within the same residue group has their
positions fixed relative to each other, and by default, every chain and
residue are in the same visibility group (default being 1). This is
undesirable when doing cross-chain design since we end up with every
designed chain being fixed in place.

Instead, we probably want every chain to be in a different residue
group, so that the model can re-position these chains as needed.
Furthermore, group 0 is intended in general for residues to be designed,
since it indicates these residues are hidden from the others.

With this understanding, we can understand why we move the full scaffold
chain above to group 2, since our target protein will be in group 1.
Furthermore, we also set all the designed chains to be in group 0.

## Query editing

Now we can download the referred scaffold structure and load it to be
edited according to the above specification.

In [ ]:
raw_scaffold_url = "https://raw.githubusercontent.com/HannesStark/boltzgen/main/example/nanobody_scaffolds/7eow.cif"
raw_scaffold_filestring = requests.get(raw_scaffold_url).text
raw_scaffold = Protein.from_string(raw_scaffold_filestring, "cif", "B")
print("raw scaffold sequence:", raw_scaffold.sequence)
print("raw scaffold coordinates shape:", raw_scaffold.coordinates.shape)
print("raw scaffold plddt shape:", raw_scaffold.plddt.shape)
print("raw scaffold name:", raw_scaffold.name)


Now let's make the edits sequentially.

In [ ]:
# mask structure at positions to design
query_scaffold = raw_scaffold.mask_structure_at(
    list(range(26, 35)) +
    list(range(52, 60)) +
    list(range(98, 119))
)
# set visibility groups
query_scaffold = query_scaffold.set_group_at(
    list(range(len(query_scaffold))), 2
)
query_scaffold = query_scaffold.set_group_at(
    list(range(26, 35)) +
    list(range(52, 60)) +
    list(range(98, 119)), 0
)
# exclude
query_scaffold = query_scaffold.delete(
    list(range(26,29)) +
    list(range(52,55)) +
    list(range(98,104))
)
# insert
# NOTE we just select a number within the range
query_scaffold = query_scaffold.batch_insert(
    {
        26: "3",
        52: "3",
        98: "7",
    }
)
# ensure insertions are group 0
query_scaffold = query_scaffold.set_group_at(
    query_scaffold.get_structure_mask(), 0
)

print("scaffold sequence:", query_scaffold.sequence)
print("scaffold structure mask:", query_scaffold.get_structure_mask().tolist())
print("scaffold groups:", Protein.get_intervals(query_scaffold._group))
print("scaffold length:", len(query_scaffold.sequence))
print("scaffold coordinates shape:", query_scaffold.coordinates.shape)
print("scaffold plddt shape:", query_scaffold.plddt.shape)


Observe that our edits made are reflected accordingly here. Note that
the structure mask has some extra residues. This is because the original
scaffold file has the structure information missing at those indices as
well. We can either drop them or just leave them here, since we have
observed that BoltzGen doesnt have issues with this.

Now let's combine the target and scaffold to form our query:

In [ ]:
# reset the chain id so they dont clash
target_protein.chain_id = "A"
query_model = target_protein & query_scaffold
print("Chains in query:", list(query_model.proteins.keys()))
print("Chain A (target chain):", query_model.proteins["A"].sequence)
print("Chain B (scaffold chain):", query_model.proteins["B"].sequence)


# Generate designs with BoltzGen

Now let's run these designs with BoltzGen:

In [ ]:
N = 1
boltzgen_job = session.models.boltzgen.generate(
    query=query_model,
    N=N,
)
boltzgen_job


Wait for completion:

In [ ]:
# This might take awhile
boltzgen_job.wait_until_done(timeout=60*60)


In [ ]:
boltzgen_job.progress_counter


In [ ]:
boltzgen_designs = boltzgen_job.get()
print("chains in design:", list(boltzgen_designs[0].proteins.keys()))
print("first design chain A sequence:", boltzgen_designs[0].proteins["A"].sequence)
print("first design chain B sequence:", boltzgen_designs[0].proteins["B"].sequence)
print("first design chain A mask:", boltzgen_designs[0].proteins["A"].get_structure_mask())
print("first design chain B mask:", boltzgen_designs[0].proteins["B"].get_structure_mask())


In [ ]:
boltzgen_designs[9].proteins["B"].get_structure_mask()


In [ ]:
# Visualize complex
builder = create_builder()
structure = (builder.download(url="complex.cif")
    .parse(format="cif")
    .model_structure()
    .component()
    .representation())
builder.molstar_notebook(data={"complex.cif": boltzgen_designs[0].make_cif_string()}, width=500, height=400)


# Inverse Folding with PoET-2 (nanobody chain)

We now propose sequences for the designed nanobody backbone using
PoET-2. We will:

- Extract the nanobody chain (binder) from the complex
- Mask its sequence for inverse folding
- Sample 10 sequences per backbone

In [ ]:
from openprotein.model import Model
from openprotein.protein import Protein

poet2_jobs = []
for i in range(N):
    generated_model = boltzgen_designs[i]
    # Mask the binder sequence to indicate that it should be generated
    generated_chain = generated_model.proteins["A"]
    query_chain = generated_chain.mask_sequence()

    # Use ProteinMPNN to design sequences for the binder backbone
    poet2_job = session.models.proteinmpnn.generate(
        query=query_chain,
        num_samples=10,
        temperature=0.1,
        seed=42, 
    )
    poet2_jobs.append(poet2_job)

# Wait for all jobs to complete
for poet2_job in poet2_jobs:
    poet2_job.wait_until_done(timeout=600)
    assert poet2_job.status == "SUCCESS"


# Structure Prediction with Boltz

We validate designed binders by predicting the complex with Boltz
(target + designed nanobody sequence). A simple loop predicts a few
designs.

In [ ]:
validated = []
# Convert the best PoET-2 sequence to a Protein and combine with the target
if len(seqs) > 0:
    # Use the first sequence
    try:
        _, seq, score = seqs[0]
    except Exception:
        # If different return shape, adapt accordingly
        seq = seqs[0][1]

    binder_prot = Protein(sequence=seq.encode())

    # For prediction, use target directly from file
    from openprotein.protein import Protein as Prot
    target_prot = Prot.from_filepath(str(TARGET_PATH), chain_id="B")

    # Run Boltz prediction for the complex
    pred_job = boltz_predict.predict(
        proteins=[binder_prot, target_prot],
        N=3,
    )
    pred_job.wait_until_done()
    preds = [pred_job.get(replicate=i) for i in range(3)]
    validated = preds

len(validated)


In [ ]:
# Visualize one predicted complex
if validated:
    builder = create_builder()
    vis = (builder.download(url="pred_complex.pdb")
        .parse(format="pdb")
        .model_structure()
        .component()
        .representation())
    builder.molstar_notebook(data={"pred_complex.pdb": validated[0]}, width=500, height=400)


# Simple filtering and ranking (optional)

You can rank candidates by simple proxy metrics (length sanity,
interface contact count if reported, model scores) before moving to more
intensive evaluation.

In [ ]:
# Placeholder for simple per-design metadata collection
# In practice, use reported scores/metrics exposed by Boltz/PoET-2 jobs.
import pandas as pd

rows = []
for i, item in enumerate(seqs[:10]):
    try:
        name, seq, score = item
        rows.append({"rank": i, "poet2_score": score, "length": len(seq)})
    except Exception:
        pass

df = pd.DataFrame(rows).sort_values(["poet2_score"], ascending=False)
df.head()


# Notes and tips

- Nanobody scaffolds: the default set includes 7eow, 7xl0, 8coh, 8z8v
  and fixes framework regions while designing CDRs (controlled via
  scaffold YAML: design, exclude, design<sub>insertions</sub>,
  reset<sub>resindex</sub>).
- Sampling: increase N in BoltzGen to broaden backbone diversity;
  increase PoET-2 num<sub>samples</sub> at low temperature for focused
  sequence search.
- Performance: large CIF uploads may trigger \>1MB warnings; expect
  longer runtimes for big N.
- API status: this tutorial uses the intended interface with multi-file
  attachments and scaffold<sub>set</sub>. Ensure your OpenProtein
  client/server has these features enabled.